# Part 1: Reinforcement Learning
Implement value iteration or policy iteration for a small discrete Markov Decision
Process (MDP).

You may use a grid world, FrozenLake, inventory control, or another small discrete environment
approved in class.


In [ ]:
import numpy as np

Clearly define the state space, action space, reward structure, and discount factor.

1. The state space is a 4x4 grid
2. the action space is the 4 directions, up, right, down, and left. There are no diagonals
3. The reward structure is comprised of a final destination which has the greatest reward of 10. A trap that has a reward of -1, so it will never be stepped in unless the agent is being impatient (low discount factor). Finally, we have a step reward of -2. Therefore, if it takes you 10 steps to get the reward, it would be the same as staying in place.
4. I went with a default discount factor of 0.9

In [ ]:
class GridWorld:
    def __init__(self):
        # the state space is a 4x4 grid represented as row, col coordinates
        self.rows = 4
        self.cols = 4

        # below are the special states
        self.start_state = (0, 0)
        self.goal_state = (3, 3)
        self.trap_state = (1, 2)
        self.obstacle_state = (2, 1)
        self.goal_state = (3, 3)

        # define the action space
        self.actions = {
            0: (-1, 0), # up
            1: (0, 1), # right
            2: (1, 0), # down
            3: (0, -1) # left
        }

        # reward structure
        self.goal_reward = 10
        self.trap_reward = -1
        self.step_reward = -2


        self.current_state = self.start_state

    def reset(self):
        self.current_state = self.start_state
        return self.current_state

    def is_terminal(self, state):
        return state == self.goal_state or state == self.trap_state

    def get_next(self, state, action):
        if self.is_terminal(state):
            return state

        row, col = state

        row_diff, col_diff = self.actions[action]

        next_row = row + row_diff
        next_col = col + col_diff

        if next_row < 0 or next_row >= self.rows or next_col < 0 or next_col >= self.cols:
            return state

        if (next_row, next_col) == self.obstacle_state:
            return state

        return (next_row, next_col)

    def get_reward(self, state):
        if state == self.goal_state:
            return self.goal_reward
        elif state == self.trap_state:
            return self.trap_reward
        else:
            return self.step_reward

Implement either <b>value iteration</b> or policy iteration and compute the final policy.


In [ ]:
def value_iteration(env, theta=0.0001, gamma=0.9):
    # initialize the value function to 0 for all states
    V = np.zeros((env.rows, env.cols))

    policy = np.zeros((env.rows, env.cols), dtype=int)

    iteration = 0

    # keep looping until it converges
    while True:
        delta = 0
        V_new = np.copy(V)

        for r in range(env.rows):
            for c in range(env.cols):
                state = (r, c)

                if env.is_terminal(state) or state == env.obstacle_state:
                    continue

                # calculate the value of all actions
                action_values = []
                for action in env.actions:
                    next_state = env.get_next(state, action)
                    reward = env.get_reward(next_state)

                    # Bellman Update
                    q_value = reward + gamma * V[next_state[0], next_state[1]]
                    action_values.append(q_value)

                # the best value will be the max q
                best_value = max(action_values)
                V_new[r, c] = best_value

                # track all the change on the grid for convergence
                delta = max(delta, np.abs(V[r, c] - best_value))
        V = V_new
        iteration += 1

        if delta < theta:
            break

    # find the final plicy using converged V
    for r in range(env.rows):
        for c in range(env.cols):
            state = (r, c)

            if env.is_terminal(state) or state == env.obstacle_state:
                continue

            action_values = []
            for action in env.actions:
                next_state = env.get_next(state, action)
                reward = env.get_reward(next_state)
                q_value = reward + gamma * V[next_state[0], next_state[1]]
                action_values.append(q_value)

            policy[r, c] = np.argmax(action_values)

    return V, policy

Show the learned value function and the final policy in a readable form.

In [ ]:
def print_values(V, env):
    print('\n\nLearned Value Function:')

    print('-' * 30)
    for r in range(env.rows):
        row_str = '|'
        for c in range(env.cols):
            state = (r,c)
            if state == env.obstacle_state:
                row_str += ' XXXX |'
            elif state == env.goal_state:
                row_str += ' GOAL |'
            elif state == env.trap_state:
                row_str += ' TRAP |'
            else:
                row_str += f' {V[r, c]:.2f} |'
        print(row_str)
        print('-' * 30)

In [ ]:
def print_policy(policy, env):
    print('\n\nLearned Policy:')

    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    print('-' * 17)

    for r in range(env.rows):
        row_str = '|'
        for c in range(env.cols):
            state = (r, c)
            if state == env.obstacle_state:
                row_str += ' X |'
            elif state == env.goal_state:
                row_str += ' G |'
            elif state == env.trap_state:
                row_str += ' T |'
            else:
                row_str += f' {arrows[policy[r, c]]} |'
        print(row_str)
        print('-' * 17)

In [ ]:
env = GridWorld()

V_optimal, policy_optimal = value_iteration(env)

print_values(V_optimal, env)
print_policy(policy_optimal, env)



Learned Value Function:
------------------------------
| -2.29 | -0.32 | 1.87 | 4.30 |
------------------------------
| -0.32 | -1.00 | TRAP | 7.00 |
------------------------------
| 1.87 | XXXX | 7.00 | 10.00 |
------------------------------
| 4.30 | 7.00 | 10.00 | GOAL |
------------------------------


Learned Policy:
-----------------
| → | → | → | ↓ |
-----------------
| ↓ | → | T | ↓ |
-----------------
| ↓ | X | → | ↓ |
-----------------
| → | → | → | G |
-----------------


Run at least one small experiment showing how the policy changes when you vary either the
discount factor or the reward design.

In [ ]:
# varying the discout factor
V_optimal, policy_optimal = value_iteration(env, gamma=0.1)

print_values(V_optimal, env)
print_policy(policy_optimal, env)



Learned Value Function:
------------------------------
| -2.21 | -2.10 | -1.00 | -2.10 |
------------------------------
| -2.10 | -1.00 | TRAP | -1.00 |
------------------------------
| -2.21 | XXXX | -1.00 | 10.00 |
------------------------------
| -2.10 | -1.00 | 10.00 | GOAL |
------------------------------


Learned Policy:
-----------------
| → | → | ↓ | ↓ |
-----------------
| → | → | T | ↓ |
-----------------
| ↑ | X | ↑ | ↓ |
-----------------
| → | → | → | G |
-----------------


Briefly discuss why this setup is an MDP and what the learned policy is doing


https://en.wikipedia.org/wiki/Markov_decision_process

This is a markov decision process because it matches the definition of the Markov Decision process as a 4-tuple. In the first part we established the state space, the action space, the transision dynamics and the rewards. Furthermore, we know this is a MDP because the GridWorld's transitions and rewards depend only upon the agents current state and imeediate action that it takes. The path it followed to get to that space is completely irrelevant, just as in all other MDPs.

We have two examples of what the learned policy is up to:
1. With gamma = 0.9: The learned policy is maximizing a longer term reward. Despite the rather harsh step cost of -2, it is still routing around the less painful trap and reaching the goal reward of +10. It is accepting shorter term penalties for a longer term reward.
2. With gamma = 0.1: The learned policy is maximizing the shorter term reward. When faced with taking another step towards and goal and inflicting a penalty of -2, or ending the run in the -1 penalty, the policy says to do the later. The agent has discovered in this situation that ending the run quickly is better than taking all those painful steps to the final goal.